# UQRoute-TC — Qwen 1.5B repeated-sample development check

**Scope:** Ten seeded generations for one clean BFCL case from the deterministic 30-case development subset. The frozen prompt and model revisions are retained. This run checks serving, saved outcomes, and canonical clustering before the broader repeated-sample experiment.

**Recorded evidence:** The manifest, ten responses, per-request seeds and timing, parse statuses, token evidence, and cluster scores are saved in Drive. The run contains no held-out cases.


## 1. Restore pinned inputs


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
root.mkdir(parents=True, exist_ok=True)
benchmark = Path('/content/robustbench-tc-release')
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
data_dir = benchmark / 'hf_data/datasets/api_eval'
assert not subprocess.check_output(['git', '-C', str(benchmark), 'status', '--porcelain', '--', str(data_dir)], text=True).strip(), 'Benchmark data has local changes'

project = Path('/content/uqroute-tc-repeated-check')
code_revision = '7c178b4c038e6af44a5c4a6599a5d3bf390e0bbf'
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'feat/task2-canonical-parser'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', code_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == code_revision
print('Benchmark:', benchmark_revision)
print('Project code:', code_revision)


## 2. Serve Qwen 1.5B on the L4


In [ ]:
import importlib.metadata
import time
import urllib.request

try:
    installed_vllm = importlib.metadata.version('vllm')
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None
packages = ['openai>=1.50,<3', '-e', str(project)]
if installed_vllm != '0.30.0':
    packages.insert(0, 'vllm==0.30.0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
import torch
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), 'Expected an L4 GPU'
assert importlib.metadata.version('vllm') == '0.30.0'

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
model_revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
endpoint = 'http://127.0.0.1:8000/v1'
server_log = Path('/content/uqroute_repeated_qwen15b_server.log')
def served_models():
    try:
        with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
            return [item['id'] for item in json.load(response)['data']]
    except Exception:
        return None

active = served_models()
if active is not None and model_id not in active:
    raise RuntimeError(f'Port 8000 is serving another model: {active}')
if active and model_id in active:
    print('Server ready:', model_id)
else:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    if f'vllm serve {model_id}' in processes:
        server = None
        print('Existing Qwen server is starting.')
    else:
        with server_log.open('w') as log:
            server = subprocess.Popen([
                'vllm', 'serve', model_id, '--revision', model_revision,
                '--tokenizer-revision', model_revision, '--dtype', 'bfloat16',
                '--max-model-len', '8192', '--gpu-memory-utilization', '0.88',
                '--port', '8000'], stdout=log, stderr=subprocess.STDOUT,
                start_new_session=True)
        print('Started server PID:', server.pid)
    for attempt in range(120):
        active = served_models()
        if active and model_id in active:
            print('Server ready:', model_id)
            break
        if server is not None and server.poll() is not None:
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
            raise RuntimeError('Server exited during startup')
        time.sleep(5)
    else:
        print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
        raise TimeoutError('Server was not ready within ten minutes')
print('GPU:', torch.cuda.get_device_name(0), '| vLLM:', importlib.metadata.version('vllm'))


## 3. Generate and save ten samples


In [ ]:
from openai import OpenAI
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import build_messages, load_samples, parse_tool_calls
from uqroute_tc.data.repeated_subset import select_repeated_subset
from uqroute_tc.inference.pilot import _atomic_json, token_evidence
from uqroute_tc.parsing.canonical import parse_prediction

split_path = project / 'docs/splits/group_split_v1.json'
split = json.loads(split_path.read_text())
assert split['benchmark_revision'] == benchmark_revision
assert hashlib.sha256(split_path.read_bytes()).hexdigest() == '688c1013d8fe9d5c1b454d14d4cbe215659e48adc7bdfd63175580e83e2eb0db'
selected = select_repeated_subset(data_dir, split)
selection_digest = hashlib.sha256(json.dumps(selected, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
assert selection_digest == '082e150679277d7d55eaa88b0b82cb236a82a32a0d6853c2cae8dc307b41fa67'
choice = next(row for row in selected if row['benchmark'] == 'bfcl_v3' and row['source_file'] == 'clean.jsonl')
sample = next(row for row in load_samples(data_dir / 'clean.jsonl') if str(row['id']) == choice['case_id'])
assert split['assignments'][choice['base_id']] == {'benchmark': 'bfcl_v3', 'population': 'primary', 'split': 'development'}

output_dir = root / 'pilots/qwen25-15b-repeated-bfcl-ten-v1'
manifest = {
    'benchmark_revision': benchmark_revision,
    'split_sha256': hashlib.sha256(split_path.read_bytes()).hexdigest(),
    'selection_digest': selection_digest,
    'code_revision': code_revision,
    'model_id': model_id, 'model_revision': model_revision,
    'tokenizer_revision': model_revision, 'engine_version': '0.30.0',
    'dtype': 'bfloat16', 'max_model_len': 8192,
    'endpoint': endpoint, 'mode': 'repeated_sampling_development_check',
    'case_id': choice['case_id'], 'source_file': choice['source_file'],
    'temperature': 0.7, 'top_p': 1.0, 'max_tokens': 1024,
    'logprobs': True, 'top_logprobs': 5,
    'sample_seeds': list(range(1729, 1739)), 'request_n': 1,
}
manifest_path = output_dir / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Saved run has different settings'
else:
    assert not output_dir.exists() or not any(output_dir.iterdir()), 'Existing output has no manifest'
    _atomic_json(manifest_path, manifest)
assert model_id in (served_models() or []), 'Qwen 1.5B server unavailable'
client = OpenAI(base_url=endpoint, api_key='EMPTY', timeout=120, max_retries=0)
for index, seed in enumerate(manifest['sample_seeds']):
    path = output_dir / 'records' / f'sample_{index:02d}.json'
    previous_attempts = []
    if path.exists():
        old = json.loads(path.read_text())
        assert old['case_id'] == choice['case_id'] and old['sample_index'] == index and old['seed'] == seed
        if old['status'] == 'complete':
            print(index, 'saved:', old['status'])
            continue
        previous_attempts = old.get('previous_attempts', []) + [
            {key: value for key, value in old.items() if key != 'previous_attempts'}
        ]
    started = time.perf_counter()
    try:
        response = client.chat.completions.create(
            model=model_id, messages=build_messages(sample),
            temperature=manifest['temperature'], top_p=manifest['top_p'],
            max_tokens=manifest['max_tokens'], logprobs=True,
            top_logprobs=manifest['top_logprobs'], seed=seed, n=1)
        response_data = response.model_dump(mode='json')
        assert len(response_data['choices']) == 1
        answer = response_data['choices'][0]
        raw = answer['message'].get('content') or ''
        evidence = token_evidence(answer)
        parsed = parse_prediction(raw, sample['benchmark'], parse_tool_calls)
        status = 'truncated' if answer.get('finish_reason') == 'length' else (
            'complete' if evidence['valid'] else 'invalid_evidence')
        error = None if status == 'complete' else (
            'generation reached max_tokens' if status == 'truncated' else evidence['reason'])
        calls = list(parsed.calls)
        finish_reason = answer.get('finish_reason')
        usage = response_data.get('usage')
    except Exception as exc:
        response_data = None
        raw, calls, finish_reason, usage = '', [], None, None
        evidence, parsed = {'valid': False, 'reason': 'request error'}, None
        status, error = 'request_error', f'{type(exc).__name__}: {exc}'
    _atomic_json(path, {
        'case_id': choice['case_id'], 'benchmark': sample['benchmark'],
        'sample_index': index, 'seed': seed, 'status': status, 'error': error,
        'previous_attempts': previous_attempts,
        'elapsed_seconds': time.perf_counter() - started,
        'finish_reason': finish_reason, 'usage': usage,
        'raw_output': raw, 'parsed_calls': calls,
        'parser_status': parsed.status if parsed else 'request_error',
        'token_evidence': evidence, 'response': response_data,
    })
    print(index, status, 'calls:', len(calls), flush=True)
print('Records:', output_dir)


## 4. Audit the saved samples


In [ ]:
from collections import Counter
from uqroute_tc.uncertainty.repeated import repeated_scores

rows = [json.loads((output_dir / 'records' / f'sample_{i:02d}.json').read_text()) for i in range(10)]
assert [r['seed'] for r in rows] == manifest['sample_seeds']
statuses = dict(Counter(r['status'] for r in rows))
parse_statuses = dict(Counter(r['parser_status'] for r in rows))
all_complete = statuses == {'complete': 10}
report = {
    'manifest': manifest, 'record_status_counts': statuses,
    'parser_status_counts': parse_statuses,
    'all_complete': all_complete,
    'elapsed_seconds_sum': sum(r['elapsed_seconds'] for r in rows),
    'usage_totals': {key: sum((r['usage'] or {}).get(key) or 0 for r in rows)
                     for key in ('prompt_tokens', 'completion_tokens', 'total_tokens')},
    'entries': [{k: r[k] for k in ('sample_index', 'seed', 'status', 'error',
                'parser_status', 'finish_reason', 'elapsed_seconds')} for r in rows],
}
if all_complete:
    assert all(r['token_evidence']['valid'] for r in rows)
    assert all(parse_tool_calls(r['raw_output'], sample['benchmark']) == r['parsed_calls'] for r in rows)
    values = repeated_scores([r['raw_output'] for r in rows], sample['benchmark'], parse_tool_calls)
    report['canonical_entropy_nats'] = values.canonical.entropy_nats
    report['canonical_disagreement'] = values.canonical.disagreement
    report['canonical_cluster_sizes'] = values.canonical.cluster_sizes
    report['exact_string_entropy_nats'] = values.exact_string.entropy_nats
else:
    report['cluster_scores_unavailable_reason'] = 'At least one request lacked complete token evidence or was truncated; statuses are retained.'
report_path = output_dir / 'repeated_check_audit.json'
_atomic_json(report_path, report)
print('Statuses:', statuses)
print('Parser statuses:', parse_statuses)
print('All complete:', all_complete)
if all_complete:
    print('Canonical cluster sizes:', values.canonical.cluster_sizes)
    print('Canonical disagreement:', values.canonical.disagreement)
print('Audit:', report_path)
